In [2]:
# --- Standard library ---
import os, re, json, uuid, textwrap, random, time
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Tuple

# --- Third party ---
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- Guard: CUDA required ---
if not torch or not torch.cuda.is_available():
    raise RuntimeError("CUDA is required. Install PyTorch with CUDA support.")

print("pandas:", pd.__version__)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0))


pandas: 2.3.3
PyTorch: 2.13.0+cu126
CUDA: True | Tesla P100-PCIE-16GB


In [3]:
from huggingface_hub import HfApi
HfApi().token = open('/home/cc/.cache/huggingface/token').read().strip()
print('HF token loaded from cache')

HF token loaded from cache


In [4]:
# --- Config ---
MODEL_NAME: str = "meta-llama/Llama-3.2-1B-Instruct"
MAX_NEW_TOKENS: int = 512
TEMPERATURE: float = 0.0          # deterministic
TRACE_DIR: Path = Path("traces")
TRACE_DIR.mkdir(exist_ok=True)
MAX_AGENT_STEPS: int = 50
MIN_EVENTS_PER_RUN: int = 90
MAX_EVENTS_PER_RUN: int = 120

# --- MCP filesystem workspace ---
WORKSPACE: Path = Path("workspace")
WORKSPACE.mkdir(exist_ok=True)

# --- Tool ID counter ---
_tool_counter = 0
def _make_tool_id() -> str:
    global _tool_counter
    _tool_counter += 1
    return f"tool_{_tool_counter:03d}"

# --- Agent definitions ---
@dataclass
class AgentConfig:
    agent_id: str
    name: str
    role: str
    description: str
    capabilities: List[str]
    system_prompt: str = ""

AGENTS: List[AgentConfig] = [
    AgentConfig(
        agent_id="agent_001", name="researcher",
        role="Senior Research Analyst",
        description="Gathers, validates, and organises information from documents, notes, and data sources. Writes summaries and reports.",
        capabilities=["read_text_file", "search_files", "list_directory", "read_multiple_files"],
        system_prompt=(
            "You are agent_001 (researcher), a Senior Research Analyst.\n"
            "You gather, validate, and organise information from documents and notes.\n"
            "When you find relevant information, write summaries to the output/ directory.\n"
            "Share findings with the analyst for calculations and interpretation.\n"
            "Use the filesystem tools to read documents, search for files, and write your findings."
        ),
    ),
    AgentConfig(
        agent_id="agent_002", name="analyst",
        role="Financial Data Analyst",
        description="Performs calculations, financial analysis, data interpretation, and synthesises findings into actionable insights.",
        capabilities=["read_text_file", "write_file", "calculator"],
        system_prompt=(
            "You are agent_002 (analyst), a Financial Data Analyst.\n"
            "You perform calculations, financial analysis, and data interpretation.\n"
            "Read documents written by the researcher, perform analysis, and write results to output/.\n"
            "Synthesise findings from multiple sources into coherent insights.\n"
            "When you have completed the analysis, call the handoff_to_researcher tool to return control."
        ),
    ),
]

AGENT_BY_ID   = {a.agent_id: a for a in AGENTS}
AGENT_BY_NAME = {a.name: a for a in AGENTS}

TASK_POOL = [
    "Read the financial report and write a summary of quarterly profits to output/financial_summary.md.",
    "Search the notes for budget planning action items and write them to output/action_items.md.",
    "Read the market analysis and calculate market share percentages. Write results to output/market_calc.md.",
    "Read the project timeline and identify the current phase. Write a status update to output/phase_status.md.",
    "Search notes for dependencies between projects and write a dependency map to output/dependencies.md.",
    "Read the financial report and calculate the ROI if we invest $500K in marketing. Write analysis to output/roi_analysis.md.",
    "Read the risk assessment document and identify the top 3 risks. Write recommendations to output/top_risks.md.",
    "Search all notes for decisions made in the last quarter. Write a decisions log to output/decisions_log.md.",
    "Calculate year-over-year growth for each product line from the financial report. Write results to output/yoy_growth.md.",
    "Read the competitive analysis and write key insights to output/competitive_insights.md.",
]

print(f"MODEL_NAME      = {MODEL_NAME}")
print(f"TRACE_DIR       = {TRACE_DIR.resolve()}")
print(f"WORKSPACE       = {WORKSPACE.resolve()}")
print(f"MAX_AGENT_STEPS = {MAX_AGENT_STEPS}")
print(f"Agents: {[a.name for a in AGENTS]}")

MODEL_NAME      = meta-llama/Llama-3.2-1B-Instruct
TRACE_DIR       = /home/cc/agent-graph/traces
WORKSPACE       = /home/cc/agent-graph/workspace
MAX_AGENT_STEPS = 50
Agents: ['researcher', 'analyst']


In [5]:
class MockLLMBackend:
    """Simulated LLM backend — no model download required.
    Generates realistic tool-calling JSON so the full agent loop works."""

    name = "mock-llm"
    model_name: str = "mock"

    def __init__(self, model_name: str):
        self.model_name = model_name
        self._global_step = 0  # increments across all calls
        print(f"MockLLM ready (no GPU/model needed).")

    def _build_prompt(self, task, history, tools, agent_name="agent"):
        # Accept all signatures — prompt building is bypassed in mock
        return ""

    def _generate(self, prompt: str) -> str:
        self._global_step += 1
        s = self._global_step
        # Use task and agent_name that were set by the last reset() call
        task = getattr(self, '_task', 'the workspace')
        agent = getattr(self, '_agent_name', 'researcher')
        tool_list = list(getattr(self, '_mcp_tools', ['list_directory']) )

        # ── phase 1: explore workspace ─────────────────────────────────────
        if s % 10 == 1:
            return json.dumps({
                "reasoning": f"First, I need to see what files are available in the workspace.",
                "action": "list_directory",
                "action_input": json.dumps({"path": "."}),
                "final_response": "",
            })

        # ── phase 2: read a relevant document ───────────────────────────────
        if s % 10 == 2:
            doc = self._pick_doc(task)
            return json.dumps({
                "reasoning": f"Found the workspace. Now I need to read {doc} to gather information.",
                "action": "read_text_file",
                "action_input": json.dumps({"path": doc}),
                "final_response": "",
            })

        # ── phase 3: search notes ───────────────────────────────────────────
        if s % 10 == 3:
            return json.dumps({
                "reasoning": "Good, I have the document. Let me also search for related notes.",
                "action": "search_files",
                "action_input": json.dumps({"path": ".", "pattern": "*.md", "excludePatterns": []}),
                "final_response": "",
            })

        # ── phase 4: list notes dir ─────────────────────────────────────────
        if s % 10 == 4:
            return json.dumps({
                "reasoning": "I have enough context now. Let me look at the notes directory.",
                "action": "list_directory",
                "action_input": json.dumps({"path": "notes"}),
                "final_response": "",
            })

        # ── phase 5: read notes ─────────────────────────────────────────────
        if s % 10 == 5:
            return json.dumps({
                "reasoning": "Now I need to read the meeting notes to find action items and decisions.",
                "action": "read_text_file",
                "action_input": json.dumps({"path": "notes/meeting_notes.md"}),
                "final_response": "",
            })

        # ── phase 6: write output ───────────────────────────────────────────
        if s % 10 == 6:
            return json.dumps({
                "reasoning": f"I have gathered enough information. Now I will write the analysis to a file.",
                "action": "write_file",
                "action_input": json.dumps({
                    "path": self._output_path(),
                    "content": self._generate_output(),
                }),
                "final_response": "",
            })

        # ── phase 7: read back to verify ────────────────────────────────────
        if s % 10 == 7:
            return json.dumps({
                "reasoning": "Let me verify the output was written correctly.",
                "action": "read_text_file",
                "action_input": json.dumps({"path": self._output_path()}),
                "final_response": "",
            })

        # ── phase 8: hand off or finish ─────────────────────────────────────
        if s % 10 == 8:
            if agent == "researcher":
                return json.dumps({
                    "reasoning": "I have gathered the information and written the initial analysis. Handing off to the analyst for deeper calculations and synthesis.",
                    "action": "handoff_to_analyst",
                    "action_input": json.dumps({}),
                    "final_response": "",
                })
            return json.dumps({
                "reasoning": "Analysis complete. All findings have been synthesised and written to the output file.",
                "action": "final",
                "action_input": json.dumps({}),
                "final_response": f"Task completed. Analyzed the data and wrote results to {self._output_path()}.",
            })

        # ── fallback: hand off if researcher, final if analyst ──────────────
        if agent == "researcher":
            return json.dumps({
                "reasoning": "Handing off to analyst for calculations.",
                "action": "handoff_to_analyst",
                "action_input": json.dumps({}),
                "final_response": "",
            })
        return json.dumps({
            "reasoning": "Task complete.",
            "action": "final",
            "action_input": json.dumps({}),
            "final_response": "Analysis complete.",
        })

    @staticmethod
    def _pick_doc(task: str) -> str:
        t = task.lower()
        if "financial" in t:        return "documents/financial_report.md"
        if "market" in t:           return "documents/market_analysis.md"
        if "project" in t or "timeline" in t: return "documents/project_timeline.md"
        if "risk" in t:             return "documents/risk_assessment.md"
        if "competitive" in t:      return "documents/competitive_analysis.md"
        if "budget" in t:           return "notes/meeting_notes.md"
        if "decision" in t:         return "notes/research_findings.md"
        return "documents/financial_report.md"

    def _output_path(self) -> str:
        t = getattr(self, '_task', '').lower()
        if "financial_summary" in t:           return "output/financial_summary.md"
        if "action_items" in t:                return "output/action_items.md"
        if "market_calc" in t:                 return "output/market_calc.md"
        if "phase_status" in t:                return "output/phase_status.md"
        if "dependencies" in t:                return "output/dependencies.md"
        if "roi" in t:                         return "output/roi_analysis.md"
        if "top_3_risks" in t or "top 3 risks" in t: return "output/top_risks.md"
        if "decisions" in t:                   return "output/decisions_log.md"
        if "year-over-year" in t or "yoy" in t: return "output/yoy_growth.md"
        if "competitive" in t:                 return "output/competitive_insights.md"
        return "output/analysis.md"

    def _generate_output(self) -> str:
        return (
            f"# Analysis Output\n\n"
            f"Task: {getattr(self, '_task', 'N/A')[:120]}\n\n"
            f"## Findings\n\n"
            f"This document was generated by the {getattr(self, '_agent_name', 'unknown')} agent using MCP filesystem tools.\n"
            f"Data sources: documents/ and notes/ directories.\n\n"
            f"## Summary\n\n"
            f"The analysis is complete. See the source documents for detailed data.\n"
        )

    def reset(self, task: str = "", agent_name: str = "researcher", mcp_tools: List[str] = None):
        self._task = task
        self._agent_name = agent_name
        self._mcp_tools = mcp_tools or []

    def _parse(self, raw: str, allowed: List[str]) -> Optional[Dict[str, Any]]:
        try:
            data = json.loads(raw)
            if "action" not in data:
                return None
            return {
                "reasoning":       str(data.get("reasoning", "")),
                "action":          str(data["action"]),
                "action_input":    str(data.get("action_input", "")),
                "final_response":  str(data.get("final_response", "")),
            }
        except (json.JSONDecodeError, KeyError):
            return None

    def decide(self, task, history, tools, agent_name="agent") -> Dict[str, Any]:
        allowed = list(tools.keys()) + ["final"]
        prompt = self._build_prompt(task, history, tools, agent_name)
        raw = self._generate(prompt)
        parsed = self._parse(raw, allowed)
        if parsed is not None:
            return parsed
        return {"reasoning": "fallback", "action": "list_directory", "action_input": '{"path": "."}', "final_response": ""}

# Replace the old LLM loading with the mock
print("Loading LLM backend…")
llm_backend = MockLLMBackend(MODEL_NAME)

Loading LLM backend…
MockLLM ready (no GPU/model needed).


In [6]:
_DOCUMENTS = {
    "financial_report": (
        "Q1 Revenue: $1,200,000 | Q2 Revenue: $1,350,000 | "
        "Q3 Revenue: $1,500,000 | Q4 Revenue: $1,800,000\n"
        "Operating Costs: $800,000 | Marketing Spend: $200,000 | R&D Investment: $300,000\n"
        "Key Milestones: Product launch in Q1, Expansion in Q3, Partnership in Q4"
    ),
    "market_analysis": (
        "Market growth rate: 15% annually | Competitors: 5 major players\n"
        "Market share: 22% | Customer satisfaction: 4.6/5\n"
        "Growth opportunities: International expansion, Product diversification"
    ),
    "project_timeline": (
        "Phase 1 (Jan–Mar): Research and Planning\n"
        "Phase 2 (Apr–Jun): Development and Testing\n"
        "Phase 3 (Jul–Sep): Marketing and Launch\n"
        "Phase 4 (Oct–Dec): Review and Optimisation"
    ),
    "risk_assessment": (
        "Financial risks: Market volatility, Currency fluctuations\n"
        "Operational risks: Supply chain delays, Resource constraints\n"
        "Strategic risks: Competition, Technology disruption\n"
        "Mitigation: Diversification, Partnerships, Innovation"
    ),
    "default": "Reference document with general information.",
}

_NOTES = [
    "Team meeting: Q1 goals review and budget allocation decisions.",
    "Research finding: AI adoption increased by 40% in the last quarter.",
    "Action item: Prepare quarterly report for board meeting.",
    "Risk alert: Supply chain disruption affecting delivery timeline.",
    "Innovation opportunity: Explore AI-powered customer support solutions.",
    "Team sync: Cross-department collaboration on new product feature.",
    "Reminder: Submit patent application for new algorithm by next month.",
    "Market insight: Competitor launched similar product — differentiate value proposition.",
]


def read_document(tool_input: str) -> Dict[str, Any]:
    key = tool_input.strip().lower()
    if "financial" in key:                 doc_id = "financial_report"
    elif "market" in key:                   doc_id = "market_analysis"
    elif "project" in key or "timeline" in key:  doc_id = "project_timeline"
    elif "risk" in key:                    doc_id = "risk_assessment"
    else:                                  doc_id = "default"
    return {"output": _DOCUMENTS[doc_id], "metadata": {"doc_id": doc_id}}


def search_notes(tool_input: str) -> Dict[str, Any]:
    query = tool_input.strip().lower()
    tokens = [t for t in re.split(r"\W+", query) if len(t) > 2]
    hits = [n for n in _NOTES if any(t in n.lower() for t in tokens)]
    return {"output": " | ".join(hits[:5] or _NOTES[:2]), "metadata": {"num_hits": len(hits)}}


def calculator(tool_input: str) -> Dict[str, Any]:
    runs = re.findall(r"[0-9.+\-*/()\s]+", tool_input)
    candidates = [r.strip() for r in runs if re.search(r"[-+*/]", r) and re.search(r"\d", r)]
    expr = max(candidates, key=len) if candidates else ""
    try:
        result = eval(expr, {"__builtins__": {}}, {}) if expr else "no expression"
    except Exception as err:
        result = f"error: {err}"
    return {"output": str(result), "metadata": {"expression": expr}}


def analyze_data(tool_input: str) -> Dict[str, Any]:
    query = tool_input.strip().lower()
    if "revenue" in query or "financial" in query:
        out = "Revenue: Q1 $1.2M, Q2 $1.35M, Q3 $1.5M, Q4 $1.8M — steady growth trajectory."
    elif "market" in query:
        out = "Market position: 22% share, 4.6/5 customer satisfaction."
    elif "growth" in query:
        out = "Growth rate: 15% annually with strong competitive positioning."
    else:
        out = "Data analysis complete. No specific patterns detected in query."
    return {"output": out, "metadata": {"query_type": query}}


def coordinate_agents(tool_input: str) -> Dict[str, Any]:
    events = [
        "Researcher shared Q4 market analysis data",
        "Analyst requested financial projections",
        "Timeline dependencies mapped between teams",
        "Cross-functional alignment on deliverables",
        "Risk assessment updated",
    ]
    return {"output": " | ".join(events), "metadata": {"num_events": len(events)}}


@dataclass
class Tool:
    tool_id: str
    name: str
    description: str
    func: Callable[[str], Dict[str, Any]]


ENHANCED_TOOLS: Dict[str, Tool] = {
    "read_document":     Tool(_make_tool_id(), "read_document",      "Read a named document (financial, market, project, risk).",      read_document),
    "search_notes":      Tool(_make_tool_id(), "search_notes",       "Search internal notes for query terms.",                       search_notes),
    "calculator":        Tool(_make_tool_id(), "calculator",         "Evaluate arithmetic expressions.",                            calculator),
    "analyze_data":      Tool(_make_tool_id(), "analyze_data",      "Analyse data patterns and trends.",                          analyze_data),
    "coordinate_agents": Tool(_make_tool_id(), "coordinate_agents", "Coordinate communication between agents.",                   coordinate_agents),
}

print("Tools registered:", list(ENHANCED_TOOLS.keys()))


Tools registered: ['read_document', 'search_notes', 'calculator', 'analyze_data', 'coordinate_agents']


## TraceCollector


`TraceEvent` carries an `execution_id` field set by `start_trace()` so both traces
in a pair share the same value. The `trace_id` is per-file: `{execution_id}a`
and `{execution_id}b`. All IDs are alphanumeric only -- no words, no underscores.


In [7]:
# --- MCP Filesystem Server ---
# The MCP session runs in a dedicated background thread with its own event loop.
# Tool calls go through a Queue bridged via call_soon_threadsafe.
# Result: persistent MCP connection across notebook cells.

import asyncio
import threading
from concurrent.futures import Future
import mcp.client.stdio as ms
from mcp import ClientSession

class MCPServer:
    """Manages an MCP filesystem server with a persistent session."""

    def __init__(self, workspace: Path):
        self.workspace = workspace
        self._tools: Dict[str, Any] = {}
        self._tool_names: List[str] = []
        self._started = threading.Event()
        self._loop: Optional[asyncio.AbstractEventLoop] = None
        self._thread: Optional[threading.Thread] = None

    def _session_loop(self):
        """Background thread: owns an event loop and the MCP session."""
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        self._loop = loop
        # Run the session in its own coroutine
        async def run():
            params = ms.StdioServerParameters(
                command="npx",
                args=["@modelcontextprotocol/server-filesystem", str(self.workspace)],
            )
            queue: asyncio.Queue = asyncio.Queue()
            self._queue = queue  # accessed by call_tool_sync via call_soon_threadsafe

            async with ms.stdio_client(params) as (read, write):
                session = ClientSession(read, write)
                async with session:
                    await session.initialize()
                    result = await session.list_tools()
                    self._tools = {t.name: t for t in result.tools}
                    self._tool_names = list(self._tools.keys())
                    self._started.set()
                    print(f"MCP server ready — {len(self._tools)} tools available")

                    # Process tool calls until stop sentinel
                    while True:
                        item = await queue.get()
                        if item is None:
                            break
                        fut, tool_name, arguments = item
                        try:
                            result = await session.call_tool(tool_name, arguments)
                            parts = []
                            for c in result.content:
                                if hasattr(c, "text") and c.text:
                                    parts.append(c.text)
                                elif hasattr(c, "data") and c.data:
                                    parts.append(str(c.data))
                            fut.set_result("\n".join(parts) if parts else str(result))
                        except Exception as e:
                            fut.set_exception(e)

        loop.run_until_complete(run())
        loop.run_forever()  # keep alive for remaining queued calls

    def start(self):
        """Launch the MCP server in a background thread."""
        if self._started.is_set():
            return self
        self._thread = threading.Thread(target=self._session_loop, daemon=True)
        self._thread.start()
        if not self._started.wait(timeout=30):
            raise RuntimeError("MCP server failed to start within 30s")
        return self

    def call_tool_sync(self, tool_name: str, arguments: Dict[str, Any]) -> str:
        """Call an MCP tool synchronously (thread-safe)."""
        if not self._started.is_set():
            raise RuntimeError("MCP server not started")
        fut = Future()
        # Schedule the put into the queue on the background thread's loop
        self._loop.call_soon_threadsafe(self._queue.put_nowait, (fut, tool_name, arguments))
        return fut.result(timeout=60)

    def get_tool_names(self) -> List[str]:
        return list(self._tool_names)

    def stop(self):
        """Shut down."""
        if self._loop and self._queue:
            self._loop.call_soon_threadsafe(self._queue.put_nowait, None)
            self._loop.call_soon_threadsafe(self._loop.stop)
        if self._thread:
            self._thread.join(timeout=5)


def start_mcp(workspace: Path) -> MCPServer:
    server = MCPServer(workspace)
    server.start()
    return server

print("MCPServer class ready.")

MCPServer class ready.


In [8]:

EVENT_TYPES = (
    "user_input", "system_init", "agent_handoff", "reasoning",
    "tool_call", "tool_result", "llm_output", "final_response",
)
SUMMARY_MAX_CHARS = 280


def _utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def _summarize(value: Any, limit: int = SUMMARY_MAX_CHARS) -> str:
    text = value if isinstance(value, str) else json.dumps(value, default=str)
    text = " ".join(text.split())
    return text if len(text) <= limit else text[: limit - 1] + "…"


@dataclass
class TraceEvent:
    """One execution event == one JSON line."""

    # --- Identity ---
    trace_id: str        # per-file unique ID  (e.g. 15924cd9a3a)
    execution_id: str     # shared between paired traces (set by start_trace)
    event_id: int
    timestamp: str
    event_type: str
    source: str
    target: str
    input_summary: str
    output_summary: str

    # --- Agent context ---
    agent_id: Optional[str] = None
    agent_name: Optional[str] = None
    agent_role: Optional[str] = None

    # --- Tool context ---
    tool_id: Optional[str] = None
    tool_name: Optional[str] = None

    # --- Behaviour ---
    expected_behavior: str = ""
    observed_behavior: str = ""

    # --- LEP fields (filled by labelling / injection step - not here) ---
    lep_injected: bool = False
    lep_type: Optional[str] = None
    lep_category: Optional[str] = None
    lep_location: Optional[str] = None
    lep_severity: Optional[str] = None

    # --- Forensic ---
    risk_tags: List[str] = field(default_factory=list)

    # --- Causality ---
    caused_by_event: Optional[int] = None
    depends_on: List[int] = field(default_factory=list)
    propagates_to: List[int] = field(default_factory=list)

    # --- Handoff ---
    agent_id_from: Optional[str] = None
    agent_name_from: Optional[str] = None
    agent_id_to: Optional[str] = None
    agent_name_to: Optional[str] = None

    # --- Failure outcome ---
    downstream_failure: bool = False
    failure_type: Optional[str] = None
    failure_event: Optional[int] = None

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


class TraceCollector:
    """Append-only JSONL trace recorder."""

    def __init__(self, trace_dir: Path = TRACE_DIR):
        self.trace_dir = Path(trace_dir)
        self.trace_dir.mkdir(exist_ok=True)
        self.execution_id: Optional[str] = None
        self.trace_id: Optional[str] = None
        self.variant: Optional[str] = None
        self.path: Optional[Path] = None
        self._next_event_id: int = 1
        self.events: List[TraceEvent] = []

    def start_trace(self, execution_id: str, variant: str) -> str:
        """Begin a new trace for one variant of an execution."""
        if variant not in ("a", "b"):
            raise ValueError("variant must be 'a' or 'b'")
        self.execution_id = execution_id
        self.variant = variant
        self.trace_id = f"{execution_id}{variant}"
        self._next_event_id = 1
        self.events = []
        self.path = self.trace_dir / f"trace_{self.trace_id}.jsonl"
        self.path.write_text("", encoding="utf-8")
        return self.trace_id

    def log_event(self, **kwargs) -> TraceEvent:
        """Create one event and append it to disk as one JSON line."""
        if self.trace_id is None or self.path is None:
            raise RuntimeError("Call start_trace() before log_event().")

        event_type = kwargs.get("event_type", "")
        if event_type not in EVENT_TYPES:
            raise ValueError(f"Unknown event_type {event_type!r}")

        event = TraceEvent(
            trace_id=self.trace_id,
            execution_id=self.execution_id,
            event_id=self._next_event_id,
            timestamp=_utc_now_iso(),
            event_type=event_type,
            source=kwargs.get("source", ""),
            target=kwargs.get("target", ""),
            input_summary=_summarize(kwargs.get("input_summary", "")),
            output_summary=_summarize(kwargs.get("output_summary", "")),
            agent_id=kwargs.get("agent_id"),
            agent_name=kwargs.get("agent_name"),
            agent_role=kwargs.get("agent_role"),
            tool_id=kwargs.get("tool_id"),
            tool_name=kwargs.get("tool_name"),
            expected_behavior=_summarize(kwargs.get("expected_behavior", "")),
            observed_behavior=_summarize(kwargs.get("observed_behavior", "")),
            agent_id_from=kwargs.get("agent_id_from"),
            agent_name_from=kwargs.get("agent_name_from"),
            agent_id_to=kwargs.get("agent_id_to"),
            agent_name_to=kwargs.get("agent_name_to"),
            lep_injected=kwargs.get("lep_injected", False),
            lep_type=kwargs.get("lep_type"),
            lep_category=kwargs.get("lep_category"),
            lep_location=kwargs.get("lep_location"),
            lep_severity=kwargs.get("lep_severity"),
            risk_tags=list(kwargs.get("risk_tags") or []),
            caused_by_event=kwargs.get("caused_by_event"),
            depends_on=list(kwargs.get("depends_on") or []),
            propagates_to=list(kwargs.get("propagates_to") or []),
            downstream_failure=kwargs.get("downstream_failure", False),
            failure_type=kwargs.get("failure_type"),
            failure_event=kwargs.get("failure_event"),
        )
        self._next_event_id += 1
        self.events.append(event)

        with self.path.open("a", encoding="utf-8") as fh:
            fh.write(json.dumps(event.to_dict(), default=str) + "\n")
        return event

    def save_jsonl(self, path: Optional[Path] = None) -> Path:
        target = Path(path) if path else self.path
        if target is None:
            raise RuntimeError("No path")
        with target.open("w", encoding="utf-8") as fh:
            for ev in self.events:
                fh.write(json.dumps(ev.to_dict(), default=str) + "\n")
        return target

    @staticmethod
    def load_jsonl(path: Path) -> List[Dict[str, Any]]:
        rows = []
        with Path(path).open("r", encoding="utf-8") as fh:
            for line in fh:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
        return rows

    def pretty_print_trace(self) -> None:
        print(f"=== Execution {self.execution_id} | Trace {self.trace_id} ({len(self.events)} events) ===")
        for ev in self.events:
            extra = ""
            if ev.agent_name: extra += f" [{ev.agent_name}]"
            if ev.tool_name:   extra += f" ->[{ev.tool_name}]"
            print(f"  [{ev.event_id:>3}] {ev.event_type:<16} {ev.source} -> {ev.target}{extra}")
            if ev.input_summary:
                print(f"        IN : {ev.input_summary[:120]}")
            if ev.output_summary:
                print(f"        OUT: {ev.output_summary[:120]}")


print("TraceCollector ready.")


TraceCollector ready.


In [9]:
# --- LEP Taxonomy (for post-hoc labelling — no injection in this notebook) ---
LEP_TAXONOMY: Dict[str, Dict[str, Any]] = {
    "FC1": {
        "name": "System Design Issues",
        "leps": {
            "1.1": "Disobey Task Specification",
            "1.2": "Disobey Role Specification",
            "1.3": "Step Repetition",
            "1.4": "Loss of Conversation History",
            "1.5": "Unaware of Termination Conditions",
        },
    },
    "FC2": {
        "name": "Inter-Agent Misalignment",
        "leps": {
            "2.1": "Conversation Reset",
            "2.2": "Fail to Ask for Clarification",
            "2.3": "Task Derailment",
            "2.4": "Information Withholding",
            "2.5": "Ignored Other Agent's Input",
            "2.6": "Reasoning-Action Mismatch",
        },
    },
    "FC3": {
        "name": "Task Verification",
        "leps": {
            "3.1": "Premature Termination",
            "3.2": "No or Incomplete Verification",
            "3.3": "Incorrect Verification",
        },
    },
}

LEP_CODE_TO_CATEGORY = {code: cat for cat, spec in LEP_TAXONOMY.items() for code in spec["leps"]}
LEP_CODE_TO_NAME     = {code: name for spec in LEP_TAXONOMY.values() for code, name in spec["leps"].items()}


def annotate_lep(events: List[Dict[str, Any]], event_id: int, lep_code: str, **kwargs) -> Dict[str, Any]:
    """Label one event with an LEP (post-hoc labelling step)."""
    if lep_code not in LEP_CODE_TO_CATEGORY:
        raise ValueError(f"Unknown LEP code: {lep_code}")
    ev = next(e for e in events if e["event_id"] == event_id)
    ev.update({
        "lep_type":     f"{lep_code} {LEP_CODE_TO_NAME[lep_code]}",
        "lep_category": LEP_CODE_TO_CATEGORY[lep_code],
        "lep_injected": True,
        **kwargs,
    })
    return ev


def link_propagation(events: List[Dict[str, Any]], cause_id: int, effect_ids: List[int]) -> None:
    """Record causal propagation edges."""
    cause = next(e for e in events if e["event_id"] == cause_id)
    cause.setdefault("propagates_to", [])
    for eid in effect_ids:
        effect = next(e for e in events if e["event_id"] == eid)
        if eid not in cause["propagates_to"]:
            cause["propagates_to"].append(eid)
        effect["caused_by_event"] = cause_id
        effect.setdefault("depends_on", [])
        if cause_id not in effect["depends_on"]:
            effect["depends_on"].append(cause_id)


print("LEP taxonomy loaded:", {cat: len(spec["leps"]) for cat, spec in LEP_TAXONOMY.items()})


LEP taxonomy loaded: {'FC1': 5, 'FC2': 6, 'FC3': 3}


In [10]:
# ═══════════════════════════════════════════════════════════════════════════
# LEP Injection Architecture
#
# Interventions operate at the action level: after the LLM produces an
# action dict (reasoning + action + action_input), the injection layer can
# modify it before the agent executes the tool.  The agent never knows it
# was interfered with — its trace shows it committing the error naturally.
#
# Available LEP codes: FC1.3, FC2.3, FC2.5, FC3.1, FC3.2
# ═══════════════════════════════════════════════════════════════════════════

from typing import Dict, List, Optional, Tuple, Callable, Any

# ── LEPInjection: a planned intervention point ──────────────────────────────

@dataclass
class LEPInjection:
    """Specifies one LEP intervention point in an execution trace."""
    lep_code: str            # e.g. "FC1.3"
    lep_name: str            # e.g. "Step Repetition"
    target_agent: str        # e.g. "researcher"
    target_step: int         # 1-indexed step number in agent loop
    trigger_event: str       # which event triggers it: "reasoning"
    params: Dict[str, Any] = field(default_factory=dict)

InterventionFn = Callable[[Dict[str, Any], LEPInjection], Dict[str, Any]]

# ── Individual intervention functions ──────────────────────────────────────

def _inject_fc13_step_repetition(action: Dict, inj: LEPInjection) -> Dict:
    """FC1.3 — Step Repetition: force agent to repeat the previous tool call."""
    modified = dict(action)
    prev_tool = inj.params.get("previous_tool", "list_directory")
    prev_args = inj.params.get("previous_args", '{"path": "."}')
    modified["action"] = prev_tool
    modified["action_input"] = prev_args
    modified["reasoning"] = (
        "Let me re-check the previous result to make sure I didn't miss "
        "anything before proceeding."
    )
    return modified


def _inject_fc25_ignored_input(action: Dict, inj: LEPInjection) -> Dict:
    """FC2.5 — Ignored Other Agent's Input: agent proceeds ignoring handoff."""
    modified = dict(action)
    modified["action"] = "list_directory"
    modified["action_input"] = '{"path": "."}'
    modified["reasoning"] = (
        "I'll continue with my own approach to gather the necessary information."
    )
    return modified


def _inject_fc31_premature_termination(action: Dict, inj: LEPInjection) -> Dict:
    """FC3.1 — Premature Termination: agent concludes before completing task."""
    return {
        "reasoning": (
            "I believe I have enough information to provide a complete answer "
            "to the user's request."
        ),
        "action": "final",
        "action_input": "",
        "final_response": inj.params.get(
            "final_response",
            "Based on my initial analysis, I have enough information. "
            "Here are the findings: relevant documents have been identified "
            "and key data points extracted. The task is complete.",
        ),
    }


def _inject_fc23_task_derailment(action: Dict, inj: LEPInjection) -> Dict:
    """FC2.3 — Task Derailment: agent diverts to an irrelevant subtask."""
    modified = dict(action)
    modified["action"] = "search_files"
    modified["action_input"] = (
        '{"path": ".", "pattern": "*.txt", "excludePatterns": []}'
    )
    modified["reasoning"] = (
        "Before continuing, I should also check for any related text files "
        "that might provide additional context for this analysis."
    )
    return modified


def _inject_fc32_no_verification(action: Dict, inj: LEPInjection) -> Dict:
    """FC3.2 — No or Incomplete Verification: agent skips verification."""
    modified = dict(action)
    if modified.get("action") in ("read_text_file", "read_multiple_files",
                                   "read_file", "list_directory"):
        modified["action"] = "final"
        modified["action_input"] = ""
        modified["final_response"] = inj.params.get(
            "final_response",
            "The output file has been successfully written. "
            "The task is complete.",
        )
        modified["reasoning"] = (
            "The write operation was successful. "
            "No further verification is necessary."
        )
    else:
        modified["reasoning"] = (
            "I'll proceed without re-reading the output file "
            "since the write was confirmed successful."
        )
    return modified


# Registry of all available LEP interventions
LEP_INTERVENTIONS: Dict[str, InterventionFn] = {
    "FC1.3": _inject_fc13_step_repetition,
    "FC2.3": _inject_fc23_task_derailment,
    "FC2.5": _inject_fc25_ignored_input,
    "FC3.1": _inject_fc31_premature_termination,
    "FC3.2": _inject_fc32_no_verification,
}


# ── LEPInjectionLayer: orchestrates planned interventions ───────────────────

class LEPInjectionLayer:
    """Manages planned LEP interventions during agent execution.

    Usage:
        layer = LEPInjectionLayer()
        layer.plan("FC1.3", "Step Repetition", "researcher", step=5)
        layer.plan("FC3.1", "Premature Termination", "analyst", step=12)

        # During each agent step:
        modified_action, meta = layer.apply("researcher", 5, original_action)
    """

    def __init__(self):
        self.injections: List[LEPInjection] = []
        self.fired: List[Dict[str, Any]] = []
        self._root_event_id: Optional[int] = None
        self._causal_chain: List[Tuple[int, int]] = []

    def plan(self, lep_code: str, lep_name: str, target_agent: str,
             target_step: int, trigger_event: str = "reasoning",
             **params) -> "LEPInjectionLayer":
        """Schedule an LEP injection at a specific (agent, step)."""
        if lep_code not in LEP_INTERVENTIONS:
            raise ValueError(
                f"Unknown LEP code: {lep_code}. "
                f"Available: {list(LEP_INTERVENTIONS.keys())}"
            )
        inj = LEPInjection(
            lep_code=lep_code,
            lep_name=lep_name,
            target_agent=target_agent,
            target_step=target_step,
            trigger_event=trigger_event,
            params=params,
        )
        self.injections.append(inj)
        return self

    def apply(self, agent_name: str, step: int,
              action: Dict[str, Any]) -> Tuple[Dict[str, Any], Optional[Dict[str, Any]]]:
        """Check if any planned intervention fires at this (agent, step).
        Returns (possibly_modified_action, metadata_or_None)."""
        for inj in self.injections:
            if (inj.target_agent == agent_name
                    and inj.target_step == step
                    and inj.trigger_event == "reasoning"):
                fn = LEP_INTERVENTIONS[inj.lep_code]
                modified = fn(action, inj)
                meta = {
                    "lep_code": inj.lep_code,
                    "lep_name": inj.lep_name,
                    "step": step,
                    "agent": agent_name,
                    "original_action": action.get("action", ""),
                    "modified_action": modified.get("action", ""),
                }
                self.fired.append(meta)
                if self._root_event_id is None:
                    self._root_event_id = step
                return modified, meta
        return action, None

    def causal_link(self, parent_event_id: int, child_event_id: int) -> None:
        """Record a causal link between LEP-affected events."""
        self._causal_chain.append((parent_event_id, child_event_id))

    @property
    def root_event_id(self) -> Optional[int]:
        return self._root_event_id

    @property
    def causal_links(self) -> List[Tuple[int, int]]:
        return list(self._causal_chain)

    def summary(self) -> Dict[str, Any]:
        """Summary of all fired interventions."""
        by_code: Dict[str, int] = {}
        for f in self.fired:
            by_code[f["lep_code"]] = by_code.get(f["lep_code"], 0) + 1
        return {
            "total_fired": len(self.fired),
            "by_code": by_code,
            "root_event_id": self._root_event_id,
            "causal_chain_length": len(self._causal_chain),
        }


# ── Standard LEP patterns (canonical intervention plans) ───────────────────
# Each pattern defines a multi-step LEP scenario with causal propagation.

def pattern_fc13_fc31_repetition_cascade(
    trigger_step: int = 5,
    agent: str = "researcher",
) -> List[LEPInjection]:
    """FC1.3 → FC3.1: Step repetition cascade leading to premature termination.
    The agent gets stuck repeating a tool call, then terminates early."""
    return [
        LEPInjection(
            lep_code="FC1.3", lep_name="Step Repetition",
            target_agent=agent, target_step=trigger_step,
            trigger_event="reasoning",
            params={"previous_tool": "read_text_file",
                    "previous_args": '{"path": "documents/financial_report.md"}'},
        ),
        LEPInjection(
            lep_code="FC1.3", lep_name="Step Repetition",
            target_agent=agent, target_step=trigger_step + 3,
            trigger_event="reasoning",
            params={"previous_tool": "search_files",
                    "previous_args": '{"path": ".", "pattern": "*.md"}'},
        ),
        LEPInjection(
            lep_code="FC3.1", lep_name="Premature Termination",
            target_agent=agent, target_step=trigger_step + 6,
            trigger_event="reasoning",
            params={"final_response": (
                "Based on my initial scan of the documents, I have gathered "
                "sufficient information. The key findings are documented. "
                "I'll finalize the output now without further verification."
            )},
        ),
    ]


def pattern_fc25_fc23_misalignment(
    trigger_step: int = 8,
    agent: str = "analyst",
) -> List[LEPInjection]:
    """FC2.5 → FC2.3: Inter-agent misalignment leading to task derailment.
    The agent ignores handoff input and derails into irrelevant searches."""
    return [
        LEPInjection(
            lep_code="FC2.5", lep_name="Ignored Other Agent's Input",
            target_agent=agent, target_step=trigger_step,
            trigger_event="reasoning",
        ),
        LEPInjection(
            lep_code="FC2.3", lep_name="Task Derailment",
            target_agent=agent, target_step=trigger_step + 4,
            trigger_event="reasoning",
        ),
        LEPInjection(
            lep_code="FC2.5", lep_name="Ignored Other Agent's Input",
            target_agent=agent, target_step=trigger_step + 8,
            trigger_event="reasoning",
            params={"final_response": (
                "I completed the analysis using my own approach. "
                "The results are in the output file."
            )},
        ),
    ]


def pattern_fc12_fc32_verification_failure(
    trigger_step: int = 6,
    agent: str = "researcher",
) -> List[LEPInjection]:
    """FC1.2 → FC3.2: Role violation leading to skipped verification.
    Agent uses wrong tools and skips verification step."""
    return [
        LEPInjection(
            lep_code="FC1.2", lep_name="Disobey Role Specification",
            target_agent=agent, target_step=trigger_step,
            trigger_event="reasoning",
            params={"final_response": (
                "The analysis is complete and verified. "
                "Output written successfully."
            )},
        ),
        LEPInjection(
            lep_code="FC3.2", lep_name="No or Incomplete Verification",
            target_agent=agent, target_step=trigger_step + 5,
            trigger_event="reasoning",
        ),
    ]


STANDARD_PATTERNS: Dict[str, Callable[[], List[LEPInjection]]] = {
    "FC1.3→FC3.1": pattern_fc13_fc31_repetition_cascade,
    "FC2.5→FC2.3": pattern_fc25_fc23_misalignment,
    "FC1.2→FC3.2": pattern_fc12_fc32_verification_failure,
}


print("LEP injection layer ready.")
print(f"  Available LEP codes: {list(LEP_INTERVENTIONS.keys())}")
print(f"  Standard patterns: {list(STANDARD_PATTERNS.keys())}")

LEP injection layer ready.
  Available LEP codes: ['FC1.3', 'FC2.3', 'FC2.5', 'FC3.1', 'FC3.2']
  Standard patterns: ['FC1.3→FC3.1', 'FC2.5→FC2.3', 'FC1.2→FC3.2']


In [11]:
# ═══════════════════════════════════════════════════════════════════════════
# MCPAgent and MultiAgentSystem
#
# MCPAgent wraps the mock LLM to build JSON-structured prompts and call
# MCP tools synchronously.  MultiAgentSystem implements a 2-agent loop
# (researcher ↔ analyst) with the LEPInjectionLayer applied at the
# reasoning step so that interventions silently modify the agent's action
# before it executes.
# ═══════════════════════════════════════════════════════════════════════════

class MCPAgent:
    """An agent that uses the MCP filesystem server for real tool calls."""

    def __init__(self, config: AgentConfig, llm_backend, mcp_server: MCPServer):
        self.config = config
        self.llm = llm_backend
        self.mcp = mcp_server
        self.history: List[Dict[str, Any]] = []

    def _build_prompt(self, task: str, handoff_context: str = "") -> str:
        tool_names = self.mcp.get_tool_names()
        tool_docs_list = []
        for name in tool_names:
            schema = self.mcp._tools.get(name)
            if schema:
                desc = getattr(schema, 'description', '') or ''
                inp = getattr(schema, 'inputSchema', {}) or {}
                props = ', '.join(inp.get('properties', {}).keys()) if inp else ''
                tool_docs_list.append(f"- {name}: {desc} (args: {props})")

        tool_docs = "\n".join(tool_docs_list)
        scratch = "\n".join(
            f"Step {i+1}: called {h['tool']}({h['args']!r}) -> {h['output'][:100]}"
            for i, h in enumerate(self.history)
        ) or "(no tool calls yet)"

        handoff_note = f"\n{handoff_context}\n" if handoff_context else ""

        prompt = (
            f"{self.config.system_prompt}\n\n"
            f"{handoff_note}"
            "You MUST use filesystem tools to accomplish the task. Do NOT reason without acting.\n"
            "Respond with ONLY a single JSON object, no prose.\n"
            "Schema: {\"reasoning\": str, \"action\": str (tool name), \"action_input\": str (JSON args), \"final_response\": str}\n"
            f"`action` must be one of: {tool_names + ['final']}\n"
            "Use 'final' only when the task is fully complete.\n"
            "IMPORTANT: action_input must be a JSON string of the arguments for the tool.\n\n"
            f"Available tools:\n{tool_docs}\n\n"
            f"Task: {task}\n\n"
            f"Your previous actions:\n{scratch}\n\n"
            "Return the JSON now."
        )
        return prompt

    def step(self, task: str, handoff_context: str = "") -> Dict[str, Any]:
        """Run one reasoning + action step. Returns the action dict."""
        if hasattr(self.llm, 'reset'):
            self.llm.reset(task=task, agent_name=self.config.name,
                           mcp_tools=self.mcp.get_tool_names())

        prompt = self._build_prompt(task, handoff_context)
        allowed = list(self.mcp._tools.keys()) + ["final"]

        raw = self.llm._generate(prompt)
        parsed = self.llm._parse(raw, allowed)

        if parsed is None:
            raw = self.llm._generate(
                f"Task: {task}\nTools: {allowed}\nRespond with JSON: "
                f'{{"reasoning": "...", "action": "tool_name", "action_input": "..."}}\n'
                f"Raw was: {raw[:200]}"
            )
            parsed = self.llm._parse(raw, allowed)
            if parsed is None:
                raise RuntimeError(f"Could not parse LLM output after retry. Raw: {raw[:300]}")

        return parsed

    def execute_tool(self, action: str, action_input: str) -> str:
        """Execute an MCP tool call synchronously and record it in history."""
        args: Dict[str, Any] = {}
        if action != "final":
            try:
                args = json.loads(action_input) if action_input else {}
            except (json.JSONDecodeError, TypeError):
                args = {"path": action_input}

        if action == "final":
            return "Task complete."

        result = self.mcp.call_tool_sync(action, args)
        self.history.append({
            "tool": action,
            "args": action_input[:200],
            "output": result[:300],
        })
        return result


class MultiAgentSystem:
    """Real multi-agent system using MCP filesystem server."""

    def __init__(self, llm_backend, mcp_server: MCPServer):
        self.llm = llm_backend
        self.mcp = mcp_server
        self.agents = {
            name: MCPAgent(cfg, llm_backend, mcp_server)
            for name, cfg in AGENT_BY_NAME.items()
        }

    def _run_execution(
        self,
        collector: TraceCollector,
        task: str,
        lep_layer: Optional[LEPInjectionLayer] = None,
    ) -> Tuple[str, int, str]:
        """Run one execution with real agent handoffs via MCP tools.
        If lep_layer is provided, interventions are applied at their
        planned (agent, step) points — the agent's actions are silently
        modified and the trace records the committed LEP.
        """
        primary   = self.agents["researcher"]
        secondary = self.agents["analyst"]
        current   = primary
        current_name = "researcher"
        handoff_count = 0
        MAX_HANDOFFS = 4
        lep_meta: Optional[Dict[str, Any]] = None

        try:
            self.mcp.call_tool_sync("create_directory", {"path": "output"})
        except Exception:
            pass

        def _agent_id(name: str) -> str:
            return f"agent_{'001' if name == 'researcher' else '002'}"

        collector.log_event(
            event_type="user_input",
            source="user", target="multi_agent_system",
            input_summary=task,
            output_summary="",
            expected_behavior="User submits a task.",
            observed_behavior=f"User task: {task[:200]}",
        )

        lep_summary = lep_layer.summary() if lep_layer else {}
        system_output = (
            f"Agents: researcher (agent_001), analyst (agent_002) "
            f"| MCP: {len(self.mcp._tools)} tools"
        )
        if lep_summary.get("total_fired"):
            system_output += (
                f" | LEPs: {lep_summary['total_fired']} intervention(s) "
                f"({', '.join(f'{k}:{v}' for k, v in lep_summary['by_code'].items())})"
            )
        collector.log_event(
            event_type="system_init",
            source="system", target="multi_agent_system",
            input_summary="Initialise multi-agent system with MCP filesystem",
            output_summary=system_output,
            expected_behavior="Both agents initialised with MCP access.",
            observed_behavior="System ready with filesystem MCP server.",
        )

        for step in range(1, MAX_AGENT_STEPS + 1):
            handoff_context = ""
            if current_name == "analyst" and step > 1:
                handoff_context = (
                    f"You received control from researcher (agent_001). "
                    f"Read their findings in output/ and continue the analysis."
                )

            action = current.step(task, handoff_context=handoff_context)

            # LEP injection intercepts the action before it executes
            if lep_layer is not None:
                action, lep_meta = lep_layer.apply(current_name, step, action)

            prev_action = action.get("action", "")

            collector.log_event(
                event_type="reasoning",
                source=_agent_id(current_name),
                target="internal",
                input_summary=f"Task: {task[:100]}",
                output_summary=action.get("reasoning", "")[:280],
                expected_behavior=f"{current.config.role} reasons toward the task goal.",
                observed_behavior=f"{current_name} reasoned: {action.get('reasoning', '')[:120]}",
                agent_id=_agent_id(current_name),
                agent_name=current_name,
                agent_role=current.config.role,
                lep_injected=(lep_meta is not None),
                lep_type=f"{lep_meta['lep_code']} {lep_meta['lep_name']}" if lep_meta else None,
                lep_category=(
                    LEP_CODE_TO_CATEGORY.get(
                        lep_meta["lep_code"].split(".")[0] + ".X", None
                    ) if lep_meta else None
                ),
                lep_location=f"step {step}",
                lep_severity=(
                    "high"
                    if lep_meta and lep_meta.get("modified_action") == "final"
                    else "medium" if lep_meta else "low"
                ),
            )

            action_type = action.get("action", "")
            action_input = action.get("action_input", "")

            # handoff: analyst -> researcher
            if action_type == "handoff_to_researcher" or (
                action_type == "final" and current_name == "analyst"
            ):
                handoff_event_id = collector.log_event(
                    event_type="agent_handoff",
                    source="agent_002", target="agent_001",
                    input_summary=f"analyst hands back to researcher after {step} steps",
                    output_summary="Control returned from analyst to researcher",
                    expected_behavior="Analyst completes analysis and returns control.",
                    observed_behavior="Handoff: agent_002 -> agent_001",
                    agent_id_from="agent_002",
                    agent_name_from="analyst",
                    agent_id_to="agent_001",
                    agent_name_to="researcher",
                ).event_id
                if lep_meta:
                    lep_layer.causal_link(
                        lep_layer.root_event_id or step, handoff_event_id
                    )
                current = primary
                current_name = "researcher"
                handoff_count += 1
                continue

            # handoff: researcher -> analyst (periodic, every 8 steps)
            if (handoff_count < MAX_HANDOFFS
                    and step > 3
                    and current_name == "researcher"
                    and step % 8 == 0
                    and action_type not in ("final", "handoff_to_analyst", "")
                    and "output" not in (action_input or "")):
                handoff_event_id = collector.log_event(
                    event_type="agent_handoff",
                    source="agent_001", target="agent_002",
                    input_summary=f"researcher hands off to analyst at step {step}",
                    output_summary="Control transferred from researcher to analyst",
                    expected_behavior="Researcher hands off to analyst for calculations.",
                    observed_behavior="Handoff: agent_001 -> agent_002",
                    agent_id_from="agent_001",
                    agent_name_from="researcher",
                    agent_id_to="agent_002",
                    agent_name_to="analyst",
                ).event_id
                if lep_meta:
                    lep_layer.causal_link(
                        lep_layer.root_event_id or step, handoff_event_id
                    )
                secondary.history = primary.history.copy()
                current = secondary
                current_name = "analyst"
                handoff_count += 1
                continue

            # final response
            if action_type == "final":
                final_response = action.get("final_response", "") or "Task complete."
                is_lep_final = lep_meta and lep_meta.get("modified_action") == "final"
                collector.log_event(
                    event_type="llm_output",
                    source=_agent_id(current_name),
                    target="internal",
                    input_summary=task,
                    output_summary=final_response[:280],
                    expected_behavior=f"{current.config.role} produces final synthesis.",
                    observed_behavior=f"{current_name} concluded.",
                    agent_id=_agent_id(current_name),
                    agent_name=current_name,
                    lep_injected=is_lep_final,
                    lep_type=f"{lep_meta['lep_code']} {lep_meta['lep_name']}" if lep_meta else None,
                    lep_category=(
                        LEP_CODE_TO_CATEGORY.get(
                            lep_meta["lep_code"].split(".")[0] + ".X", None
                        ) if lep_meta else None
                    ),
                    lep_location=f"step {step}",
                    lep_severity="high" if is_lep_final else "low",
                    downstream_failure=is_lep_final,
                    failure_type="premature_termination" if is_lep_final else None,
                    failure_event=collector.events[-1].event_id if is_lep_final else None,
                )
                final_event_id = collector.log_event(
                    event_type="final_response",
                    source=_agent_id(current_name),
                    target="user",
                    input_summary=task,
                    output_summary=final_response[:280],
                    expected_behavior="Answer returned to user.",
                    observed_behavior=f"Done. {handoff_count} handoffs, {len(current.history)} tool calls by {current_name}.",
                    agent_id=_agent_id(current_name),
                    agent_name=current_name,
                    lep_injected=is_lep_final,
                    lep_type=f"{lep_meta['lep_code']} {lep_meta['lep_name']}" if lep_meta else None,
                    lep_category=(
                        LEP_CODE_TO_CATEGORY.get(
                            lep_meta["lep_code"].split(".")[0] + ".X", None
                        ) if lep_meta else None
                    ),
                    lep_location=f"step {step}",
                    lep_severity="high" if is_lep_final else "low",
                    downstream_failure=is_lep_final,
                    failure_type="premature_termination" if is_lep_final else None,
                    failure_event=collector.events[-1].event_id - 1 if is_lep_final else None,
                ).event_id
                if lep_meta:
                    lep_layer.causal_link(
                        lep_layer.root_event_id or step, final_event_id
                    )
                return collector.trace_id, len(collector.events), final_response

            if not action_type or action_type not in list(self.mcp._tools.keys()):
                continue

            # tool call via MCP
            tool_schema = self.mcp._tools.get(action_type)
            tool_desc = getattr(tool_schema, 'description', action_type) if tool_schema else action_type
            is_lep_tool = lep_meta and lep_meta.get("modified_action") == action_type

            tool_call_event_id = collector.log_event(
                event_type="tool_call",
                source=_agent_id(current_name),
                target=f"mcp_{action_type}",
                input_summary=action_input[:280],
                output_summary="",
                expected_behavior=f"{current_name} invokes '{action_type}' via MCP.",
                observed_behavior=f"{current_name} called mcp_{action_type} ({tool_desc}).",
                agent_id=_agent_id(current_name),
                agent_name=current_name,
                tool_id=f"mcp_{action_type}",
                tool_name=action_type,
                lep_injected=is_lep_tool,
                lep_type=f"{lep_meta['lep_code']} {lep_meta['lep_name']}" if lep_meta else None,
                lep_category=(
                    LEP_CODE_TO_CATEGORY.get(
                        lep_meta["lep_code"].split(".")[0] + ".X", None
                    ) if lep_meta else None
                ),
                lep_location=f"step {step}",
                lep_severity="medium" if is_lep_tool else "low",
            ).event_id

            try:
                tool_output = current.execute_tool(action_type, action_input)
            except Exception as e:
                tool_output = f"Error: {e}"

            is_lep_result = is_lep_tool and lep_meta.get("lep_code", "").startswith("FC1.3")

            tool_result_event_id = collector.log_event(
                event_type="tool_result",
                source=f"mcp_{action_type}",
                target=_agent_id(current_name),
                input_summary=f"{action_type} input: {action_input[:100]}",
                output_summary=tool_output[:280],
                expected_behavior=f"MCP tool '{action_type}' returns result.",
                observed_behavior=f"MCP returned {len(tool_output)} chars.",
                agent_id=_agent_id(current_name),
                agent_name=current_name,
                tool_id=f"mcp_{action_type}",
                tool_name=action_type,
                lep_injected=is_lep_result,
                lep_type=f"{lep_meta['lep_code']} {lep_meta['lep_name']}" if lep_meta else None,
                lep_category=(
                    LEP_CODE_TO_CATEGORY.get(
                        lep_meta["lep_code"].split(".")[0] + ".X", None
                    ) if lep_meta else None
                ),
                lep_location=f"step {step}",
                lep_severity="medium" if is_lep_result else "low",
            ).event_id

            if is_lep_tool and lep_layer:
                lep_layer.causal_link(
                    lep_layer.root_event_id or step, tool_call_event_id
                )
                lep_layer.causal_link(tool_call_event_id, tool_result_event_id)

            time.sleep(0.05)

        final_response = f"Max steps ({MAX_AGENT_STEPS}) reached. {handoff_count} handoffs."
        collector.log_event(
            event_type="final_response",
            source="agent_001", target="user",
            input_summary=task,
            output_summary=final_response,
            expected_behavior="Partial results at step limit.",
            observed_behavior="Max steps reached.",
            agent_id="agent_001",
            agent_name="researcher",
        )
        return collector.trace_id, len(collector.events), final_response

    def generate_execution_pair(
        self,
        execution_id: str,
        task: str,
        lep_layer: Optional[LEPInjectionLayer] = None,
    ) -> Dict[str, Any]:
        """Generate both variants of one execution."""
        coll_a = TraceCollector()
        trace_id_a = coll_a.start_trace(execution_id, "a")
        tid_a, n_a, resp_a = self._run_execution(coll_a, task, lep_layer=None)

        output_path = WORKSPACE / "output"
        if output_path.exists():
            import shutil
            shutil.rmtree(output_path, ignore_errors=True)
            output_path.mkdir(exist_ok=True)

        coll_b = TraceCollector()
        trace_id_b = coll_b.start_trace(execution_id, "b")
        tid_b, n_b, resp_b = self._run_execution(coll_b, task, lep_layer=lep_layer)

        return {
            "execution_id": execution_id,
            "task": task,
            "variant_a": {
                "trace_id": tid_a, "path": str(coll_a.path),
                "num_events": n_a, "final_response": resp_a,
            },
            "variant_b": {
                "trace_id": tid_b, "path": str(coll_b.path),
                "num_events": n_b, "final_response": resp_b,
            },
        }

    def generate_dataset(
        self,
        num_executions: int = 3,
        lep_layer: Optional[LEPInjectionLayer] = None,
    ) -> Dict[str, Any]:
        """Generate `num_executions` paired traces."""
        executions = []
        for i in range(num_executions):
            execution_id = uuid.uuid4().hex[:10]
            task = random.choice(TASK_POOL)
            result = self.generate_execution_pair(
                execution_id, task, lep_layer=lep_layer
            )
            executions.append(result)
            print(
                f"  [{execution_id}]"
                f"  A={result['variant_a']['num_events']} events"
                f"  B={result['variant_b']['num_events']} events"
                f"  — {task[:55]}…"
            )

        metadata = {
            "num_executions": num_executions,
            "agents": [{"agent_id": a.agent_id, "name": a.name, "role": a.role} for a in AGENTS],
            "mcp_tools": self.mcp.get_tool_names(),
            "min_events_per_trace": MIN_EVENTS_PER_RUN,
            "max_events_per_trace": MAX_EVENTS_PER_RUN,
            "generated_at": datetime.now(timezone.utc).isoformat(),
        }

        meta_path = TRACE_DIR / "dataset_metadata.json"
        with meta_path.open("w", encoding="utf-8") as f:
            json.dump(metadata, f, indent=2)

        summary_path = TRACE_DIR / "dataset_summary.json"
        total_a = sum(e["variant_a"]["num_events"] for e in executions)
        total_b = sum(e["variant_b"]["num_events"] for e in executions)
        summary = {
            "num_executions": num_executions,
            "total_traces": num_executions * 2,
            "total_events": total_a + total_b,
            "avg_events_per_trace": (total_a + total_b) / (num_executions * 2),
            "trace_dir": str(TRACE_DIR.resolve()),
            "generated_at": datetime.now(timezone.utc).isoformat(),
        }
        with summary_path.open("w", encoding="utf-8") as f:
            json.dump(summary, f, indent=2)

        print(f"\nMetadata  -> {meta_path}")
        print(f"Summary   -> {summary_path}")
        return {"executions": executions, "metadata": metadata, "summary": summary}

    def generate_dataset_with_lep(
        self,
        num_executions: int = 3,
        pattern: str = "FC1.3->FC3.1",
    ) -> Dict[str, Any]:
        """Generate paired traces with a standard LEP pattern on variant_b."""
        if pattern not in STANDARD_PATTERNS:
            raise ValueError(
                f"Unknown pattern: {pattern}. "
                f"Available: {list(STANDARD_PATTERNS.keys())}"
            )
        injections = STANDARD_PATTERNS[pattern]()
        lep_layer = LEPInjectionLayer()
        for inj in injections:
            lep_layer.injections.append(inj)
        print(
            f"LEP pattern '{pattern}' loaded: "
            f"{len(injections)} intervention(s) planned\n"
            + "\n".join(
                f"  {inj.lep_code} {inj.lep_name} -> "
                f"{inj.target_agent} @ step {inj.target_step}"
                for inj in injections
            )
        )
        return self.generate_dataset(
            num_executions=num_executions, lep_layer=lep_layer
        )


print("MultiAgentSystem with MCP ready.")

MultiAgentSystem with MCP ready.


In [12]:
# --- Start MCP server and run multi-agent traces ---
print("Starting MCP filesystem server…")
mcp_server = start_mcp(WORKSPACE)
print(f"MCP tools: {mcp_server.get_tool_names()}\n")

mas = MultiAgentSystem(llm_backend, mcp_server)
print("Starting trace generation…\n")
dataset = mas.generate_dataset(num_executions=3)

Starting MCP filesystem server…
MCP server ready — 14 tools availableMCP tools: ['read_file', 'read_text_file', 'read_media_file', 'read_multiple_files', 'write_file', 'edit_file', 'create_directory', 'list_directory', 'list_directory_with_sizes', 'directory_tree', 'move_file', 'search_files', 'get_file_info', 'list_allowed_directories']

Starting trace generation…


  [361408b3ef]  A=123 events  B=123 events  — Read the market analysis and calculate market share per…
  [2e7fa04c6c]  A=123 events  B=123 events  — Read the competitive analysis and write key insights to…
  [b4947fbe0d]  A=123 events  B=123 events  — Calculate year-over-year growth for each product line f…

Metadata  -> traces/dataset_metadata.json
Summary   -> traces/dataset_summary.json


In [13]:
# --- Verification & Analysis ---
print("\n=== Verifying generated traces ===\n")

all_traces = []
for exec_data in dataset["executions"]:
    for variant_key in ("variant_a", "variant_b"):
        v = exec_data[variant_key]
        path = Path(v["path"])
        if path.exists():
            events = TraceCollector.load_jsonl(path)
            all_traces.append({
                "execution_id": exec_data["execution_id"],
                "trace_id": v["trace_id"],
                "variant": variant_key.split("_")[1],
                "num_events": len(events),
                "events": events,
            })
            # Confirm execution_id is consistent across both variants
            exec_ids_in_file = {e["execution_id"] for e in events}
            assert exec_ids_in_file == {exec_data["execution_id"]}, \
                f"execution_id mismatch in {v['trace_id']}"
            print(f"  \u2713 {v['trace_id']}  ({len(events)} events)  execution_id={exec_data['execution_id']}")
        else:
            print(f"  \u2717 MISSING: {path}")

print(f"\n{len(all_traces)} trace files verified.\n")

# ── Build DataFrame ──────────────────────────────────────────────────────────
rows = []
for tr in all_traces:
    for ev in tr["events"]:
        rows.append({
            "execution_id": tr["execution_id"],
            "trace_id": tr["trace_id"],
            "variant": tr["variant"],
            "event_id": ev["event_id"],
            "event_type": ev["event_type"],
            "source": ev["source"],
            "target": ev["target"],
            "agent_id": ev.get("agent_id", ""),
            "agent_name": ev.get("agent_name", ""),
            "tool_id": ev.get("tool_id", ""),
            "tool_name": ev.get("tool_name", ""),
            "input_summary": ev.get("input_summary", "")[:80],
            "output_summary": ev.get("output_summary", "")[:80],
        })

df = pd.DataFrame(rows)
print(f"DataFrame: {len(df)} rows \u00d7 {len(df.columns)} columns\n")

# ── Event counts by execution / variant ─────────────────────────────────────
print("=== Events per trace ===")
print(df.groupby(["execution_id", "variant"])["event_id"].count().rename("num_events").reset_index())

# ── Event type distribution ────────────────────────────────────────────────
print("\n=== Event type distribution ===")
print(df.pivot_table(index="event_type", columns="variant", aggfunc="size", fill_value=0))

# ── Agent activity ─────────────────────────────────────────────────────────
print("\n=== Tool calls per agent ===")
tool_calls = df[df["event_type"] == "tool_call"]
print(tool_calls.groupby(["agent_name", "tool_name"]).size().rename("count").reset_index())

# ── Handoff counts ─────────────────────────────────────────────────────────
print("\n=== Agent handoffs per execution ===")
handoffs = df[df["event_type"] == "agent_handoff"]
print(handoffs.groupby(["execution_id", "variant"]).size().rename("handoffs").reset_index())

# ── Save processed CSV ─────────────────────────────────────────────────────
csv_path = TRACE_DIR / "processed_dataset.csv"
df.to_csv(csv_path, index=False)
print(f"\nProcessed CSV saved \u2192 {csv_path}")



=== Verifying generated traces ===

  ✓ 361408b3efa  (123 events)  execution_id=361408b3ef
  ✓ 361408b3efb  (123 events)  execution_id=361408b3ef
  ✓ 2e7fa04c6ca  (123 events)  execution_id=2e7fa04c6c
  ✓ 2e7fa04c6cb  (123 events)  execution_id=2e7fa04c6c
  ✓ b4947fbe0da  (123 events)  execution_id=b4947fbe0d
  ✓ b4947fbe0db  (123 events)  execution_id=b4947fbe0d

6 trace files verified.

DataFrame: 738 rows × 13 columns

=== Events per trace ===
  execution_id variant  num_events
0   2e7fa04c6c       a         123
1   2e7fa04c6c       b         123
2   361408b3ef       a         123
3   361408b3ef       b         123
4   b4947fbe0d       a         123
5   b4947fbe0d       b         123

=== Event type distribution ===
variant           a    b
event_type              
agent_handoff    12   12
final_response    3    3
reasoning       150  150
system_init       3    3
tool_call        99   99
tool_result      99   99
user_input        3    3

=== Tool calls per agent ===
   agent_name  

In [14]:
# --- Pretty-print one example trace ---
example_trace_id = all_traces[0]["trace_id"]
example_events  = all_traces[0]["events"]

print(f"=== Sample trace: {example_trace_id} ===\n")
for ev in example_events[:20]:
    agent_str = f" [{ev.get('agent_name', '')}]" if ev.get("agent_name") else ""
    tool_str  = f" ->[{ev.get('tool_name', '')}]"  if ev.get("tool_name")  else ""
    print(f"  #{ev['event_id']:>3}  {ev['event_type']:<16}  {ev['source']} -> {ev['target']}  {agent_str}{tool_str}")
    if ev.get("input_summary"):
        print(f"         IN : {ev['input_summary'][:100]}")
    if ev.get("output_summary"):
        print(f"         OUT: {ev['output_summary'][:100]}")

=== Sample trace: 361408b3efa ===

  #  1  user_input        user -> multi_agent_system  
         IN : Read the market analysis and calculate market share percentages. Write results to output/market_calc
  #  2  system_init       system -> multi_agent_system  
         IN : Initialise multi-agent system with MCP filesystem
         OUT: Agents: researcher (agent_001), analyst (agent_002) | MCP: 14 tools
  #  3  reasoning         agent_001 -> internal   [researcher]
         IN : Task: Read the market analysis and calculate market share percentages. Write results to output/marke
         OUT: First, I need to see what files are available in the workspace.
  #  4  tool_call         agent_001 -> mcp_list_directory   [researcher] ->[list_directory]
         IN : {"path": "."}
  #  5  tool_result       mcp_list_directory -> agent_001   [researcher] ->[list_directory]
         IN : list_directory input: {"path": "."}
         OUT: [DIR] documents [DIR] notes [DIR] output [FILE] readme.txt
 

In [15]:
# --- Confirm both files share the same execution_id ---
exec_example = dataset["executions"][0]["execution_id"]
a_events = TraceCollector.load_jsonl(Path(dataset["executions"][0]["variant_a"]["path"]))
b_events = TraceCollector.load_jsonl(Path(dataset["executions"][0]["variant_b"]["path"]))

a_exec_ids = {e["execution_id"] for e in a_events}
b_exec_ids = {e["execution_id"] for e in b_events}

print(f"Execution: {exec_example}")
print(f"  variant_a execution_ids: {a_exec_ids}")
print(f"  variant_b execution_ids: {b_exec_ids}")
assert a_exec_ids == b_exec_ids == {exec_example}, "execution_id mismatch between variants!"
print("  \u2713 Both variants share the same execution_id \u2014 pairing is correct.")

# --- List all generated trace files ---
print("\n=== Generated trace files ===\n")
for fpath in sorted(TRACE_DIR.glob("trace_*.jsonl")):
    size_kb = fpath.stat().st_size / 1024
    print(f"  {fpath.name}  ({size_kb:.1f} KB)")


Execution: 361408b3ef
  variant_a execution_ids: {'361408b3ef'}
  variant_b execution_ids: {'361408b3ef'}
  ✓ Both variants share the same execution_id — pairing is correct.

=== Generated trace files ===

  trace_02e9040b30a.jsonl  (131.3 KB)
  trace_02e9040b30b.jsonl  (132.2 KB)
  trace_2e7fa04c6ca.jsonl  (131.9 KB)
  trace_2e7fa04c6cb.jsonl  (131.9 KB)
  trace_361408b3efa.jsonl  (132.5 KB)
  trace_361408b3efb.jsonl  (132.5 KB)
  trace_5d22b89957a.jsonl  (131.9 KB)
  trace_5d22b89957b.jsonl  (131.9 KB)
  trace_9fb4fc35aca.jsonl  (131.8 KB)
  trace_9fb4fc35acb.jsonl  (132.4 KB)
  trace_b4947fbe0da.jsonl  (132.4 KB)
  trace_b4947fbe0db.jsonl  (132.4 KB)


In [16]:
# --- LEP Injection: label malignant (variant-b) traces in-place ---
# Three distinct LEP patterns, one per execution pair.
# Overwrites the .jsonl file for variant-b — no separate _labeled file.

from collections import Counter

LEP_CODE_TO_CATEGORY = {code: cat for cat, spec in LEP_TAXONOMY.items() for code in spec["leps"]}
LEP_CODE_TO_NAME     = {code: name for spec in LEP_TAXONOMY.values() for code, name in spec["leps"].items()}


def _apply_lep(events, event_id, lep_code, **kwargs):
    if lep_code not in LEP_CODE_TO_CATEGORY:
        raise ValueError(f"Unknown LEP code: {lep_code}")
    ev = next(e for e in events if e["event_id"] == event_id)
    ev.update({
        "lep_type":     f"{lep_code} {LEP_CODE_TO_NAME[lep_code]}",
        "lep_category": LEP_CODE_TO_CATEGORY[lep_code],
        "lep_injected": True,
        **kwargs,
    })
    return ev


def _link(events, cause_id, effect_ids):
    cause = next(e for e in events if e["event_id"] == cause_id)
    cause.setdefault("propagates_to", [])
    for eid in effect_ids:
        effect = next(e for e in events if e["event_id"] == eid)
        if eid not in cause["propagates_to"]:
            cause["propagates_to"].append(eid)
        effect["caused_by_event"] = cause_id
        effect.setdefault("depends_on", [])
        if cause_id not in effect["depends_on"]:
            effect["depends_on"].append(cause_id)


def _save(events, path):
    with Path(path).open("w", encoding="utf-8") as fh:
        for e in events:
            fh.write(json.dumps(e, default=str) + "\n")


# ─── pattern 1: repetition cascade → premature termination ─────────────────
def _inject_pattern_repetition(events):
    tool_calls   = [e for e in events if e["event_type"] == "tool_call"]
    reasoning_evs = [e for e in events if e["event_type"] == "reasoning"]
    final_ev     = next((e for e in events if e["event_type"] == "final_response"), None)
    tool_targets = Counter(e["target"] for e in tool_calls)
    repeated     = {t for t, c in tool_targets.items() if c >= 3}

    seen, rep_ids = set(), []
    for tc in tool_calls:
        if tc["target"] in repeated:
            if tc["target"] in seen:
                _apply_lep(events, tc["event_id"], "1.3",
                           lep_location="agent loop", lep_severity="medium")
                rep_ids.append(tc["event_id"])
            seen.add(tc["target"])

    # 2 downstream reasoning events contaminated
    reason_after = [re for re in reasoning_evs if rep_ids and re["event_id"] > rep_ids[0]]
    for re_ev in reason_after[:2]:
        _apply_lep(events, re_ev["event_id"], "1.3",
                   lep_location="downstream reasoning", lep_severity="low")
        _link(events, rep_ids[0], [re_ev["event_id"]])

    if final_ev:
        _apply_lep(events, final_ev["event_id"], "3.1",
                   lep_location="termination", lep_severity="high",
                   downstream_failure=True, failure_type="premature_termination",
                   failure_event=final_ev["event_id"])
        if rep_ids:
            _link(events, rep_ids[0], [final_ev["event_id"]])
        return rep_ids[0] if rep_ids else None
    return None


# ─── pattern 2: inter-agent misalignment → ignored handoff ──────────────────
def _inject_pattern_misalignment(events):
    tool_calls   = [e for e in events if e["event_type"] == "tool_call"]
    handoffs     = [e for e in events if e["event_type"] == "agent_handoff"]
    reasoning_evs = [e for e in events if e["event_type"] == "reasoning"]
    final_ev     = next((e for e in events if e["event_type"] == "final_response"), None)
    a1_calls     = [tc for tc in tool_calls if tc["source"] == "agent_001"]

    handoff_ev = handoffs[0] if handoffs else None
    if handoff_ev:
        ignored_tool = a1_calls[0]["target"] if a1_calls else None
        stuck_calls  = [tc for tc in tool_calls if tc["target"] == ignored_tool]
        if len(stuck_calls) >= 2:
            for tc in stuck_calls[1:]:
                _apply_lep(events, tc["event_id"], "2.5",
                           lep_location="agent handoff boundary", lep_severity="high")
                _link(events, handoff_ev["event_id"], [tc["event_id"]])

    if reasoning_evs:
        mid = reasoning_evs[len(reasoning_evs) // 2]
        _apply_lep(events, mid["event_id"], "2.3",
                   lep_location="reasoning step", lep_severity="medium")
        if handoff_ev:
            _link(events, handoff_ev["event_id"], [mid["event_id"]])

    if final_ev:
        _apply_lep(events, final_ev["event_id"], "2.5",
                   lep_location="final response", lep_severity="high",
                   downstream_failure=True, failure_type="incomplete_synthesis",
                   failure_event=final_ev["event_id"])
        if handoff_ev:
            _link(events, handoff_ev["event_id"], [final_ev["event_id"]])
        return handoff_ev["event_id"] if handoff_ev else None
    return None


# ─── pattern 3: verification failure → unsafe tool usage ────────────────────
def _inject_pattern_verification(events):
    tool_calls = [e for e in events if e["event_type"] == "tool_call"]
    final_ev   = next((e for e in events if e["event_type"] == "final_response"), None)

    rogue_calls = [tc for tc in tool_calls
                   if tc["source"] == "agent_001" and tc["target"] == "tool_003"]
    if rogue_calls:
        rc = rogue_calls[0]
        _apply_lep(events, rc["event_id"], "1.2",
                   lep_location="tool selection", lep_severity="high")
        _apply_lep(events, rc["event_id"], "3.2",
                   lep_location="tool selection", lep_severity="high")

        corr = next(
            (e for e in events if e["event_type"] == "tool_result"
             and e["event_id"] == rc["event_id"] + 1), None
        )
        if corr:
            corr["downstream_failure"] = True
            corr["failure_type"]       = "unsafe_tool_usage"
            corr["failure_event"]      = rc["event_id"]
            _link(events, rc["event_id"], [corr["event_id"]])

        reason_after = [e for e in events
                        if e["event_type"] == "reasoning" and e["event_id"] > rc["event_id"]]
        for re_ev in reason_after[:2]:
            _apply_lep(events, re_ev["event_id"], "3.2",
                       lep_location="downstream reasoning", lep_severity="medium")
            _link(events, rc["event_id"], [re_ev["event_id"]])

    if final_ev:
        _apply_lep(events, final_ev["event_id"], "3.2",
                   lep_location="final response", lep_severity="high",
                   downstream_failure=True, failure_type="hallucinated_answer",
                   failure_event=final_ev["event_id"])
        if rogue_calls:
            _link(events, rogue_calls[0]["event_id"], [final_ev["event_id"]])
        return rogue_calls[0]["event_id"] if rogue_calls else None
    return None


# ─── run across all 3 pairs ────────────────────────────────────────────────

PATTERNS = [
    ("FC1.3 → FC3.1", "Repetition cascade → premature termination",    _inject_pattern_repetition),
    ("FC2.5 → FC2.3", "Inter-agent misalignment → ignored handoff",      _inject_pattern_misalignment),
    ("FC1.2 → FC3.2", "Role violation → verification failure",           _inject_pattern_verification),
]

print(f"{'#':<2}  {'execution_id':<12}  {'trace_id':<14}  lep_events  root_lep    failure_type            targets")
print("─" * 110)

summary_rows = []
for idx, exec_data in enumerate(dataset["executions"]):
    code, desc, fn = PATTERNS[idx]

    b_path = Path(exec_data["variant_b"]["path"])
    events = TraceCollector.load_jsonl(b_path)
    root   = fn(events)               # mutate in-place
    _save(events, b_path)             # overwrite variant-b file

    lep_evs  = [e for e in events if e.get("lep_injected")]
    failures = [e for e in events if e.get("downstream_failure")]
    root_code = next((e["lep_type"].split(" ")[0] for e in lep_evs), "")
    ftype     = failures[0]["failure_type"] if failures else "—"
    targets   = sorted({t for e in lep_evs for t in (e.get("propagates_to") or [])})

    summary_rows.append({
        "execution_id":      exec_data["execution_id"],
        "trace_id":          exec_data["variant_b"]["trace_id"],
        "pattern":           code,
        "description":       desc,
        "lep_events":        len(lep_evs),
        "root_lep":          root_code,
        "failure_type":      ftype,
        "propagation_depth": max(targets) - root if targets and root else 0,
        "total_events":      len(events),
    })

    print(f"{idx:<2}  {exec_data['execution_id']:<12}  {exec_data['variant_b']['trace_id']:<14}"
          f"  {len(lep_evs):<11}  {root_code:<11}  {ftype:<22}  {targets}")

# ─── summary ────────────────────────────────────────────────────────────────
print(f"\n{'=' * 90}")
print("=== LEP Injection Summary ===")
for row in summary_rows:
    print(f"\n  Execution   : {row['execution_id']}  (trace {row['trace_id']})")
    print(f"  Pattern     : {row['pattern']} — {row['description']}")
    print(f"  LEP events  : {row['lep_events']} / {row['total_events']}")
    print(f"  Root LEP    : {row['root_lep']}")
    print(f"  Failure     : {row['failure_type']}")
    print(f"  Prop. depth : {row['propagation_depth']} steps")

print("\n✓ All 3 malignant traces labeled in-place (no _labeled files created).")

#   execution_id  trace_id        lep_events  root_lep    failure_type            targets
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
0   361408b3ef    361408b3efb     32           1.3          premature_termination   [15, 18, 123]
1   2e7fa04c6c    2e7fa04c6cb     10           2.5          incomplete_synthesis    []
2   b4947fbe0d    b4947fbe0db     1            3.2          hallucinated_answer     []

=== LEP Injection Summary ===

  Execution   : 361408b3ef  (trace 361408b3efb)
  Pattern     : FC1.3 → FC3.1 — Repetition cascade → premature termination
  LEP events  : 32 / 123
  Root LEP    : 1.3
  Failure     : premature_termination
  Prop. depth : 110 steps

  Execution   : 2e7fa04c6c  (trace 2e7fa04c6cb)
  Pattern     : FC2.5 → FC2.3 — Inter-agent misalignment → ignored handoff
  LEP events  : 10 / 123
  Root LEP    : 2.5
  Failure     : incomplete_synthesis
  Prop. depth : 0 steps

  Execution   : b4947fbe0d  (tra